# Phase B — Deep Mel CNN (2-class direction) — Google Colab

Train the Phase B transfer model on **GPU**, with all artifacts on **Google Drive**:

- `best.pt` — best validation-loss weights
- `last.pt` — full resume state (model + optimizer + scheduler + history)
- `best_history.json`, `train_summary.json`, `run_config.json`

**Resume behaviour**
- End-of-epoch: `last.pt` written after every epoch (`batch_idx=0`).
- Mid-epoch: `last.pt` also written every `CHECKPOINT_EVERY_BATCHES` training batches.
- After a Colab disconnect: re-run all cells; training auto-detects `last.pt` on Drive and continues (including mid-epoch).

**Prerequisites on Drive**
1. Upload or sync **IDMT-Traffic** dataset (folder must contain `audio/` and `annotation/`).
2. Upload this repo (or clone from Git) — the notebook installs `idmt_experiments` from `IDMT_experiments/src`.

## 1. Mount Drive & install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q librosa soundfile scikit-learn tqdm pandas

## 2. Configuration (edit paths here)

In [ ]:
from pathlib import Path

# --- Google Drive layout ---
DRIVE_ROOT = Path('/content/drive/MyDrive/DopplerLab')

# IDMT-Traffic root (must contain audio/ + annotation/)
DATA_DIR = DRIVE_ROOT / 'IDMT_Traffic'

# All checkpoints & outputs live on Drive (survive runtime restarts)
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
OUTPUT_DIR = DRIVE_ROOT / 'outputs'

# --- Training ---
RUN_NAME = 'deep_mel_2class_left_100ep'   # folder under transfer/direction/
MONO_SOURCE = 'left'                       # 'left' or 'right'
EPOCHS = 100
PREEMPT = False                            # True = early stopping
CHECKPOINT_EVERY_BATCHES = 25              # mid-epoch resume granularity
AUTO_RESUME = True                         # continue from last.pt if present

# Repo: clone once to Drive, or upload IDMT_experiments folder
REPO_DIR = DRIVE_ROOT / 'DopplerLab'       # git clone target
USE_GIT_CLONE = False                      # set True first time to clone repo
GIT_URL = 'https://github.com/YOUR_ORG/DopplerLab.git'  # update if using clone

RUN_DIR = CHECKPOINT_DIR / 'transfer' / 'direction' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Run directory:', RUN_DIR)
print('Data directory:', DATA_DIR)

## 3. Install `idmt_experiments` package

In [ ]:
import sys
import subprocess

if USE_GIT_CLONE and not (REPO_DIR / 'IDMT_experiments').exists():
    subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, str(REPO_DIR)], check=True)

# Fallback: copy from Colab upload at /content/DopplerLab
SRC_CANDIDATES = [
    REPO_DIR / 'IDMT_experiments' / 'src',
    Path('/content/DopplerLab/IDMT_experiments/src'),
    Path('/content/IDMT_experiments/src'),
]
PKG_SRC = next((p for p in SRC_CANDIDATES if (p / 'idmt_experiments').exists()), None)
if PKG_SRC is None:
    raise FileNotFoundError(
        'Cannot find idmt_experiments. Set REPO_DIR or upload IDMT_experiments/src to /content/'
    )

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PKG_SRC)], check=True)
print('Installed from', PKG_SRC)

In [ ]:
import json
import idmt_experiments.config as cfg_mod
from idmt_experiments.config import DirectionConfig
from idmt_experiments.cnn.train_recipe import phase_a_recipe
from idmt_experiments.engine.train_mel import train_mel_split
from idmt_experiments.transfer.model import build_model
from idmt_experiments.transfer.eval import run_eval
from idmt_experiments.training_resume import last_state_path, load_training_state
from idmt_experiments.src.preprocess import resolve_data_dir

# Point package defaults at Drive
cfg_mod.DEFAULT_CHECKPOINT_DIR = CHECKPOINT_DIR
cfg_mod.DEFAULT_OUTPUT_DIR = OUTPUT_DIR
cfg_mod.DEFAULT_DATA_DIR = DATA_DIR

data_root = resolve_data_dir(DATA_DIR)
assert (data_root / 'audio').is_dir(), f'Missing audio/: {data_root}'
print('Dataset OK:', data_root)

## 4. Resume status (read before training)

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt_path = RUN_DIR / 'best.pt'
last_path = last_state_path(ckpt_path)

resume = AUTO_RESUME and last_path.exists()
if resume:
    st = load_training_state(ckpt_path, device)
    if st.mid_epoch:
        print(f'Will resume MID-EPOCH {st.epoch}, batch {st.batch_idx}')
    else:
        print(f'Will resume after completed epoch {st.epoch} -> start epoch {st.epoch + 1}')
    print(f'  best_epoch={st.best_epoch}  history_len={len(st.history)}  target_epochs={st.epochs_configured}')
    if st.history:
        last = st.history[-1]
        print(f'  last logged: epoch {last["epoch"]} val_bal={last.get("val_bal_acc", 0):.4f}')
else:
    print('Fresh training (no last.pt on Drive)')

print('Device:', device)

## 5. Train Phase B (100 epochs, 2-class L2R/R2L)

In [ ]:
cfg = DirectionConfig(
    task='direction',
    feature_type='mel',
    mono_source=MONO_SOURCE,
    n_classes=2,
    epochs=EPOCHS,
    preempt=PREEMPT,
    weight_decay=1e-3,
    patience=15,
)
recipe = phase_a_recipe()

print('=' * 72)
print('PHASE B - Deep Mel CNN')
print('=' * 72)

ckpt = train_mel_split(
    run_name=RUN_NAME,
    cfg=cfg,
    model_factory=lambda c: build_model(c, 'deep_mel_cnn'),
    checkpoint_subdir='transfer/direction',
    checkpoint_dir=CHECKPOINT_DIR,
    data_dir=DATA_DIR,
    device=device,
    resume=resume,
    train_recipe=recipe,
    extra_ckpt={'backbone': 'deep_mel_cnn', 'phase': 'B', 'colab': True},
    checkpoint_every_batches=CHECKPOINT_EVERY_BATCHES,
)
print('Checkpoint:', ckpt)

## 6. Training history & metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

hist_path = RUN_DIR / 'best_history.json'
if hist_path.exists():
    history = json.loads(hist_path.read_text())
    df = pd.DataFrame(history)
    display(df.tail(10))
    if 'val_bal_acc' in df.columns:
        ax = df.plot(x='epoch', y=['val_bal_acc', 'train_acc'], figsize=(8, 4), title='Phase B learning curve')
        ax.set_ylabel('accuracy')
        plt.show()
        print('Best val balanced acc:', df['val_bal_acc'].max())
else:
    print('No history file yet:', hist_path)

## 7. Evaluate on test split (vehicle 2-class)

In [ ]:
out_eval = run_eval(
    ckpt,
    data_dir=DATA_DIR,
    output_subdir='transfer/direction',
    device=device,
)
metrics = json.loads((out_eval / 'eval_metrics.json').read_text())
print('Test balanced accuracy:', metrics['balanced_accuracy'])
print('Per-class recall:', metrics.get('per_class_recall'))
print('Flip agreement:', metrics.get('flip_agreement'))
print('Saved to', out_eval)

## 8. Download weights to local machine

Files are already on Drive. To copy to your PC, download:

- `best.pt` — use for inference / Phase C fusion
- `last.pt` — resume training
- `run_config.json`, `train_summary.json`, `best_history.json`

Local path after download: place under `IDMT_experiments/checkpoints/transfer/direction/<RUN_NAME>/`

In [ ]:
from google.colab import files

for name in ['best.pt', 'last.pt', 'train_summary.json', 'best_history.json', 'run_config.json']:
    p = RUN_DIR / name
    if p.exists():
        print('Downloading', p)
        files.download(str(p))
    else:
        print('Skip (missing):', p)

## Colab tips

1. **Runtime disconnected?** Re-run from section 1. `AUTO_RESUME=True` picks up `last.pt` on Drive.
2. **Extend training:** increase `EPOCHS` and re-run section 5 (resume uses new target).
3. **Right channel:** set `MONO_SOURCE='right'` and a new `RUN_NAME`.
4. **Faster restarts:** precomputed mel is rebuilt each session (~1–2 min on GPU); checkpoints avoid retraining epochs.